# COMP5318 Assignment 1: Rice Classification

##### Group number: 69
##### Student 1 SID: 540207680
##### Student 2 SID: 530181464

## Table of Contents

- [1. Data Pre-processing](#1-data-pre-processing)
- [2. Build Classifiers](#2-build-classifiers)
  - [Part 1: Cross-validation without parameter tuning](#part-1-cross-validation-without-parameter-tuning)
  - [Part 2: Cross-validation with parameter tuning](#part-2-cross-validation-with-parameter-tuning)
  - [Part 2: Results](#part-2-results)
- [3. Reflection and Discussion](#3-reflection-and-discussion)


## **1. Data Pre-processing**

In [1]:
# Import all libraries
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [3]:
# Load the rice dataset from the assignment folder
data_path = Path("rice-final2.csv")

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path.resolve()}")

riceDataFrame = pd.read_csv(data_path, na_values="?")

if riceDataFrame.shape[1] < 2:
    raise ValueError("The dataset must contain at least one feature column and one class column.")

X = riceDataFrame.iloc[:, :-1].copy()
y = riceDataFrame.iloc[:, -1].copy()

print("Dataset shape:", riceDataFrame.shape)
print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("Class values:", sorted(y.astype(str).str.strip().unique()))


Dataset shape: (1400, 8)
Feature matrix shape: (1400, 7)
Target vector shape: (1400,)
Class values: ['class2' 'class1']


In [4]:
# Pre-process the dataset in a reusable way so it works with other files of the same format.
def preprocess_dataset(X, y):
    """Fill missing values, normalize features, and encode class labels."""
    if X.empty:
        raise ValueError("Feature matrix is empty.")
    if y.empty:
        raise ValueError("Target vector is empty.")

    # Convert feature columns to numeric so non-numeric placeholders become NaN.
    X_numeric = X.apply(pd.to_numeric, errors="coerce")

    # Guard against columns that contain no usable values at all.
    fully_missing_cols = X_numeric.columns[X_numeric.isna().all()].tolist()
    if fully_missing_cols:
        raise ValueError(f"These feature columns contain only missing or invalid values: {fully_missing_cols}")

    # Replace missing values with the mean of each column.
    imputer = SimpleImputer(strategy="mean")
    X_imputed = imputer.fit_transform(X_numeric)

    # Scale each feature to the [0, 1] range.
    scaler = MinMaxScaler()
    X_processed = scaler.fit_transform(X_imputed)

    # Encode class labels class1 -> 0 and class2 -> 1.
    y_clean = y.astype(str).str.strip()
    label_map = {"class1": 0, "class2": 1}
    y_processed = y_clean.map(label_map)

    if y_processed.isnull().any():
        unknown_labels = sorted(y_clean[y_processed.isnull()].unique())
        raise ValueError(f"Unexpected class labels found: {unknown_labels}")

    return X_processed, y_processed.astype(int).to_numpy(), imputer, scaler

XProcessed, yProcessed, imputer, scaler = preprocess_dataset(X, y)


In [5]:
# Print the first ten rows of the pre-processed dataset using four decimal places.
def print_data(X, y, n_rows=10):
    """Print the first n_rows of a numpy feature matrix and label vector."""
    if len(X) != len(y):
        raise ValueError("Feature matrix and target vector must have the same number of rows.")

    rows_to_print = min(n_rows, len(X))
    for example_num in range(rows_to_print):
        feature_text = ",".join(f"{feature:.4f}" for feature in X[example_num])
        print(f"{feature_text},{int(y[example_num])}")

print_data(XProcessed, yProcessed)


0.4628,0.5406,0.5113,0.4803,0.7380,0.4699,0.1196,1
0.4900,0.5547,0.5266,0.5018,0.7319,0.4926,0.8030,1
0.6109,0.6847,0.6707,0.5409,0.8032,0.6253,0.1185,0
0.6466,0.6930,0.6677,0.5961,0.7601,0.6467,0.2669,0
0.6712,0.6233,0.4755,0.8293,0.3721,0.6803,0.4211,1
0.2634,0.2932,0.2414,0.4127,0.5521,0.2752,0.2825,1
0.8175,0.9501,0.9515,0.5925,0.9245,0.8162,0.0000,0
0.3174,0.3588,0.3601,0.3908,0.6921,0.3261,0.8510,1
0.3130,0.3050,0.2150,0.5189,0.3974,0.3159,0.4570,1
0.5120,0.5237,0.4409,0.6235,0.5460,0.5111,0.3155,1


In [19]:
# Display the first ten rows as a formatted table for a cleaner notebook preview.
from IPython.display import display

def preview_data_table(X, y, n_rows=10):
    """Display the first n_rows as a styled table without changing the preprocessing logic."""
    if len(X) != len(y):
        raise ValueError("Feature matrix and target vector must have the same number of rows.")

    rows_to_show = min(n_rows, len(X))
    feature_cols = [f"Feature {i + 1}" for i in range(X.shape[1])]
    preview_df = pd.DataFrame(X[:rows_to_show], columns=feature_cols)
    preview_df["Class"] = y[:rows_to_show].astype(int)

    styled = (
        preview_df.style
        .format({col: "{:.4f}" for col in feature_cols})
        .set_caption("Pre-processed Dataset Preview")
        .set_table_styles([
            {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "16px"), ("font-weight", "600"), ("color", "#111827")]},
            {"selector": "th", "props": [("background-color", "#1f2937"), ("color", "white"), ("text-align", "center"), ("padding", "8px 10px")]},
            {"selector": "td", "props": [("text-align", "center"), ("padding", "8px 10px")]},
            {"selector": "table", "props": [("border-collapse", "collapse"), ("border", "1px solid #d1d5db"), ("border-radius", "8px"), ("overflow", "hidden")]},
        ])
    )
    display(styled)

preview_data_table(XProcessed, yProcessed)


,Feature 1,Feature 2,Feature 3,Feature 4,Feature 5,Feature 6,Feature 7,Class
0,0.4628,0.5406,0.5113,0.4803,0.7380,0.4699,0.1196,1
1,0.4900,0.5547,0.5266,0.5018,0.7319,0.4926,0.8030,1
2,0.6109,0.6847,0.6707,0.5409,0.8032,0.6253,0.1185,0
3,0.6466,0.6930,0.6677,0.5961,0.7601,0.6467,0.2669,0
4,0.6712,0.6233,0.4755,0.8293,0.3721,0.6803,0.4211,1
5,0.2634,0.2932,0.2414,0.4127,0.5521,0.2752,0.2825,1
6,0.8175,0.9501,0.9515,0.5925,0.9245,0.8162,0.0000,0
7,0.3174,0.3588,0.3601,0.3908,0.6921,0.3261,0.8510,1
8,0.3130,0.3050,0.2150,0.5189,0.3974,0.3159,0.4570,1
9,0.5120,0.5237,0.4409,0.6235,0.5460,0.5111,0.3155,1


## **2. Build Classifiers**

- Part 1:  Logistic Regression, Naïve Bayes
- Part 2:  KNN, Decision Tree, Ada Boost, Gradient Boost, Random Forest, SVM

### Part 1: Cross-validation without parameter tuning

In [6]:
## Setting the 10 fold stratified cross-validation
cvKFold=StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# The stratified folds from cvKFold should be provided to the classifiers

In [7]:
# Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

logRModel = LogisticRegression(
    max_iter=1000,
    random_state=0
)

logRScores = cross_val_score(
    estimator=logRModel,
    X=XProcessed,
    y=yProcessed,
    cv=cvKFold,
    scoring="accuracy"
)

logRAvgAccuracy = logRScores.mean()

In [8]:
# Naïve Bayes
from sklearn.naive_bayes import GaussianNB

nbModel = GaussianNB()

nbScores = cross_val_score(
    estimator=nbModel,
    X=XProcessed,
    y=yProcessed,
    cv=cvKFold,
    scoring="accuracy"
)

nbAvgAccuracy = nbScores.mean()

### Part 1 Results


In [9]:
# Print results for each classifier in part 1 to 4 decimal places here:
print(f"LogR average cross-validation accuracy: {logRAvgAccuracy:.4f}")
print(f"NB average cross-validation accuracy: {nbAvgAccuracy:.4f}")

LogR average cross-validation accuracy: 0.9386
NB average cross-validation accuracy: 0.9264


### Part 2: Cross-validation with parameter tuning

In [10]:
# Split the data into training and test sets first so the same split can be reused by Decision Tree, AdaBoost, Gradient Boosting, Random Forest, and SVM.
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

xTrain, xTest, yTrain, yTest = train_test_split(
    XProcessed,
    yProcessed,
    test_size=0.2,
    stratify=yProcessed,
    random_state=0
)


In [11]:
# KNN
# parameters you may consider
k = [1, 3, 5, 7]
p = [1, 2]

# p=1 -> Manhattan distance
# p=2 -> Euclidean distance
knnParameterGrid = {
    "n_neighbors": k,
    "p": p
}

knnModel = KNeighborsClassifier(metric="minkowski")

knnGridSearch = GridSearchCV(
    estimator=knnModel,
    param_grid=knnParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=-1
)

knnGridSearch.fit(xTrain, yTrain)

knnBestModel = knnGridSearch.best_estimator_
knnBestK = knnGridSearch.best_params_["n_neighbors"]
knnBestP = knnGridSearch.best_params_["p"]
knnBestCvAccuracy = knnGridSearch.best_score_

knnTestPrediction = knnBestModel.predict(xTest)
knnTestAccuracy = accuracy_score(yTest, knnTestPrediction)

In [12]:
# Decision Tree
# parameters you may consider
max_depth = [3, 5, 7, 10]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

decisionTreeParameterGrid = {
    "max_depth": max_depth,
    "min_samples_split": min_samples_split,
    "min_samples_leaf": min_samples_leaf
}

decisionTreeModel = DecisionTreeClassifier(random_state=0)

decisionTreeGridSearch = GridSearchCV(
    estimator=decisionTreeModel,
    param_grid=decisionTreeParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=-1
)

decisionTreeGridSearch.fit(xTrain, yTrain)

decisionTreeBestModel = decisionTreeGridSearch.best_estimator_
decisionTreeBestMaxDepth = decisionTreeGridSearch.best_params_["max_depth"]
decisionTreeBestMinSamplesSplit = decisionTreeGridSearch.best_params_["min_samples_split"]
decisionTreeBestMinSamplesLeaf = decisionTreeGridSearch.best_params_["min_samples_leaf"]
decisionTreeBestCvAccuracy = decisionTreeGridSearch.best_score_

decisionTreeTestPrediction = decisionTreeBestModel.predict(xTest)
decisionTreeTestAccuracy = accuracy_score(yTest, decisionTreeTestPrediction)

In [13]:
# Ada Boost
# parameters you may consider
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score

adaBoostParameterGrid = {
    "n_estimators": n_estimators,
    "learning_rate": learning_rate
}

adaBoostModel = AdaBoostClassifier(random_state=0)

adaBoostGridSearch = GridSearchCV(
    estimator=adaBoostModel,
    param_grid=adaBoostParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=-1
)

adaBoostGridSearch.fit(xTrain, yTrain)

adaBoostTestPrediction = adaBoostGridSearch.best_estimator_.predict(xTest)
adaBoostTestAccuracy = accuracy_score(yTest, adaBoostTestPrediction)

In [14]:
# Gradient Boost
# parameters you may consider
max_depth = [1, 3, 5, 7]
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

gradientBoostParameterGrid = {
    "max_depth": max_depth,
    "n_estimators": n_estimators,
    "learning_rate": learning_rate
}

gradientBoostModel = GradientBoostingClassifier(random_state=0)

gradientBoostGridSearch = GridSearchCV(
    estimator=gradientBoostModel,
    param_grid=gradientBoostParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=-1
)

gradientBoostGridSearch.fit(xTrain, yTrain)

gradientBoostTestPrediction = gradientBoostGridSearch.best_estimator_.predict(xTest)
gradientBoostTestAccuracy = accuracy_score(yTest, gradientBoostTestPrediction)

In [15]:
# Random Forest
# You should use RandomForestClassifier from sklearn.ensemble with information gain and max_features set to 'sqrt'.
# parameters you may consider
n_estimators = [10, 30, 60, 100]
max_leaf_nodes = [6, 12]

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

randomForestParameterGrid = {
    "n_estimators": n_estimators,
    "max_leaf_nodes": max_leaf_nodes
}

randomForestModel = RandomForestClassifier(
    criterion="entropy",      # information gain
    max_features="sqrt",
    random_state=0
)

randomForestGridSearch = GridSearchCV(
    estimator=randomForestModel,
    param_grid=randomForestParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=-1
)

randomForestGridSearch.fit(xTrain, yTrain)

randomForestTestPrediction = randomForestGridSearch.best_estimator_.predict(xTest)
randomForestTestAccuracy = accuracy_score(yTest, randomForestTestPrediction)
randomForestMacroF1 = f1_score(yTest, randomForestTestPrediction, average="macro")
randomForestWeightedF1 = f1_score(yTest, randomForestTestPrediction, average="weighted")

In [16]:
# SVM
# parameters you may consider
C = [0.01, 0.1, 1, 5]
gamma = [0.01, 0.1, 1, 10]

# optional
kernel = ["rbf"]

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

svmParameterGrid = {
    "C": C,
    "gamma": gamma,
    "kernel": kernel
}

svmModel = SVC(random_state=0)

svmGridSearch = GridSearchCV(
    estimator=svmModel,
    param_grid=svmParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=-1
)

svmGridSearch.fit(xTrain, yTrain)

svmTestPrediction = svmGridSearch.best_estimator_.predict(xTest)
svmTestAccuracy = accuracy_score(yTest, svmTestPrediction)


### Part 2: Results

In [17]:
# Perform Grid Search with 10-fold stratified cross-validation (GridSearchCV in sklearn). 
# The stratified folds from cvKFold should be provided to GridSearchV

# This should include using train_test_split from sklearn.model_selection with stratification and random_state=0
# Print results for each classifier here. All the reported results should be printed to 4 decimal places except for the integers such as "k", "p", n_estimators" and "max_leaf_nodes".

# example printing:
# Print results for each classifier here.
# All the reported results should be printed to 4 decimal places except for integers.

print(f"KNN best k: {knnGridSearch.best_params_['n_neighbors']}")
print(f"KNN best p: {knnGridSearch.best_params_['p']}")
print(f"KNN cross-validation accuracy: {knnGridSearch.best_score_:.4f}")
print(f"KNN test set accuracy: {knnTestAccuracy:.4f}")
print()

print(f"Decision Tree best max_depth: {decisionTreeGridSearch.best_params_['max_depth']}")
print(f"Decision Tree best min_samples_split: {decisionTreeGridSearch.best_params_['min_samples_split']}")
print(f"Decision Tree best min_samples_leaf: {decisionTreeGridSearch.best_params_['min_samples_leaf']}")
print(f"Decision Tree cross-validation accuracy: {decisionTreeGridSearch.best_score_:.4f}")
print(f"Decision Tree test set accuracy: {decisionTreeTestAccuracy:.4f}")
print()

print(f"AdaBoost best n_estimators: {adaBoostGridSearch.best_params_['n_estimators']}")
print(f"AdaBoost best learning_rate: {adaBoostGridSearch.best_params_['learning_rate']:.4f}")
print(f"AdaBoost cross-validation accuracy: {adaBoostGridSearch.best_score_:.4f}")
print(f"AdaBoost test set accuracy: {adaBoostTestAccuracy:.4f}")
print()

print(f"Gradient Boost best max_depth: {gradientBoostGridSearch.best_params_['max_depth']}")
print(f"Gradient Boost best n_estimators: {gradientBoostGridSearch.best_params_['n_estimators']}")
print(f"Gradient Boost best learning_rate: {gradientBoostGridSearch.best_params_['learning_rate']:.4f}")
print(f"Gradient Boost cross-validation accuracy: {gradientBoostGridSearch.best_score_:.4f}")
print(f"Gradient Boost test set accuracy: {gradientBoostTestAccuracy:.4f}")
print()

print(f"RF best n_estimators: {randomForestGridSearch.best_params_['n_estimators']}")
print(f"RF best max_leaf_nodes: {randomForestGridSearch.best_params_['max_leaf_nodes']}")
print(f"RF cross-validation accuracy: {randomForestGridSearch.best_score_:.4f}")
print(f"RF test set accuracy: {randomForestTestAccuracy:.4f}")
print(f"RF test set macro average F1: {randomForestMacroF1:.4f}")
print(f"RF test set weighted average F1: {randomForestWeightedF1:.4f}")
print()

print(f"SVM best C: {svmGridSearch.best_params_['C']}")
print(f"SVM best gamma: {svmGridSearch.best_params_['gamma']:.4f}")
print(f"SVM best kernel: {svmGridSearch.best_params_['kernel']}")
print(f"SVM cross-validation accuracy: {svmGridSearch.best_score_:.4f}")
print(f"SVM test set accuracy: {svmTestAccuracy:.4f}")

KNN best k: 7
KNN best p: 2
KNN cross-validation accuracy: 0.9375
KNN test set accuracy: 0.9250

Decision Tree best max_depth: 3
Decision Tree best min_samples_split: 2
Decision Tree best min_samples_leaf: 1
Decision Tree cross-validation accuracy: 0.9357
Decision Tree test set accuracy: 0.9429

AdaBoost best n_estimators: 100
AdaBoost best learning_rate: 0.1000
AdaBoost cross-validation accuracy: 0.9437
AdaBoost test set accuracy: 0.9393

Gradient Boost best max_depth: 1
Gradient Boost best n_estimators: 50
Gradient Boost best learning_rate: 0.1000
Gradient Boost cross-validation accuracy: 0.9446
Gradient Boost test set accuracy: 0.9429

RF best n_estimators: 30
RF best max_leaf_nodes: 6
RF cross-validation accuracy: 0.9411
RF test set accuracy: 0.9429
RF test set macro average F1: 0.9414
RF test set weighted average F1: 0.9427

SVM best C: 5
SVM best gamma: 1.0000
SVM best kernel: rbf
SVM cross-validation accuracy: 0.9429
SVM test set accuracy: 0.9321


### Test your code

In [18]:
# Load the test dataset to test out your model
from sklearn.metrics import accuracy_score, f1_score

testFileName = "test-before.csv"

# Read dataset
testDf = pd.read_csv(testFileName, na_values='?')

# Separate features and labels
xExternalRaw = testDf.iloc[:, :-1]
yExternalRaw = testDf.iloc[:, -1]

# Convert labels
yExternal = yExternalRaw.replace({"class1": 0, "class2": 1}).astype(int)

print("External test dataset loaded successfully.")
print(f"Dataset shape: {testDf.shape}")
print(f"Feature matrix shape: {xExternalRaw.shape}")
print(f"Target vector shape: {yExternal.shape}")

# Apply the same preprocessing learned from the rice dataset
xExternalImputed = imputer.transform(xExternalRaw)
xExternalScaled = scaler.transform(xExternalImputed)

# Choose the already trained models
allModels = {
    "Logistic Regression": lrModel,
    "Naive Bayes": nbModel,
    "KNN": knnGridSearch.best_estimator_,
    "Decision Tree": dtGridSearch.best_estimator_,
    "AdaBoost": adaGridSearch.best_estimator_,
    "Gradient Boosting": gbGridSearch.best_estimator_,
    "Random Forest": rfGridSearch.best_estimator_,
    "SVM": svmGridSearch.best_estimator_
}

print("\nExternal test results on test-before.csv")
print("-" * 60)

for modelName, model in allModels.items():
    yExternalPred = model.predict(xExternalScaled)

    externalAccuracy = accuracy_score(yExternal, yExternalPred)
    externalMacroF1 = f1_score(yExternal, yExternalPred, average="macro")
    externalWeightedF1 = f1_score(yExternal, yExternalPred, average="weighted")

    print(f"{modelName} external test accuracy: {externalAccuracy:.4f}")
    print(f"{modelName} external test macro average F1: {externalMacroF1:.4f}")
    print(f"{modelName} external test weighted average F1: {externalWeightedF1:.4f}")
    print()

External test dataset loaded successfully.
Dataset shape: (209, 7)
Feature matrix shape: (209, 6)
Target vector shape: (209,)


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- a1
- a2
- a3
- a4
- a5
- ...
Feature names seen at fit time, yet now missing:
- Area
- Convex_Area
- Eccentricity
- Extent
- Major_Axis_Length
- ...


## **3. Reflection and Discussion**

